# ⚡ Imagen → 3D CON TEXTURA — Stable-Fast-3D (Stability AI, gratis)

Genera un modelo 3D **ya texturizado** (con los colores de tu imagen) en ~1 segundo de GPU.
Usa solo **~6 GB de VRAM** → **entra bien en la T4 gratis** (a diferencia de la textura de Hunyuan que moría por memoria).

## Requisito único (una sola vez, gratis)
Stability publica el modelo con acceso "gated" en Hugging Face:
1. Creá una cuenta gratis en https://huggingface.co (si no tenés).
2. Entrá a **https://huggingface.co/stabilityai/stable-fast-3d** y tocá **"Agree and access repository"** (licencia comunitaria: gratis para uso personal/no comercial).
3. Creá un token en **https://huggingface.co/settings/tokens** → "New token" → tipo **Read** → copialo (empieza con `hf_`).

## Orden
1. **GPU T4**: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.
2. **Celda 1** (instalar, ~4-6 min) → **Celda 1B** (pegar tu token hf_).
3. **Celda 2** (subir tu imagen, mejor PNG sin fondo — igual quita el fondo solo).
4. **Celda 3** (generar) → **Celda 4** (descargar `.glb` texturizado).

## Celda 1 — Instalar Stable-Fast-3D
Compila dos extensiones pequeñas (texture_baker y uv_unwrapper); tarda unos minutos. Ignorá los warnings amarillos.

In [ ]:
!nvidia-smi -L

import os
os.chdir('/content')
if not os.path.isdir('/content/stable-fast-3d'):
    !git clone https://github.com/Stability-AI/stable-fast-3d.git
os.chdir('/content/stable-fast-3d')

!pip install -q "setuptools==69.5.1" wheel
!pip install -q -r requirements.txt 2>&1 | tail -3
# extensiones nativas (si requirements ya las instalo, esto no hace nada)
!pip install -q ./texture_baker ./uv_unwrapper 2>&1 | tail -2

import torch
print('\ntorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print('✅ Celda 1 lista. Seguí con la Celda 1B (token).')

## Celda 1B — Tu token de Hugging Face
Pegalo cuando te lo pida (no queda guardado en el notebook). Antes tenés que haber aceptado la licencia en la página del modelo (ver arriba).

In [ ]:
from getpass import getpass
from huggingface_hub import login
tok = getpass('Pegá tu token hf_... y Enter: ')
login(token=tok.strip())
print('✅ Sesión de Hugging Face iniciada. Seguí con la Celda 2.')

## Celda 2 — Subir tu imagen
Cualquier imagen del personaje **de frente**. SF3D le quita el fondo solo (rembg), pero un PNG ya recortado da mejor resultado.
Si el botón de subir no anda (celular): subila por el panel **Archivos** 📁 (carpeta de la izquierda) y corré esta celda igual — la detecta sola.

In [ ]:
import os, glob
from PIL import Image

IMG = None
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMG = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget no anduvo (', e, ') -> uso el panel Archivos.')

if not IMG or not os.path.exists(IMG):
    cand = []
    for ext in ('png','jpg','jpeg','webp'):
        cand += glob.glob('/content/*.'+ext)
    cand.sort(key=os.path.getmtime)
    IMG = cand[-1] if cand else None

assert IMG and os.path.exists(IMG), 'No encontré imagen. Subila por el botón o por el panel Archivos 📁 y volvé a correr esta celda.'
im = Image.open(IMG)
print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)
print('✅ Lista. Seguí con la Celda 3.')

## Celda 3 — Generar el modelo 3D texturizado
La primera vez baja el modelo (~2 GB). Después genera en segundos. Sale `mesh.glb` **con textura** (UV + material).

In [ ]:
import os
os.chdir('/content/stable-fast-3d')
os.makedirs('output', exist_ok=True)

!python run.py "{IMG}" --output-dir output/ --texture-resolution 1024

# run.py guarda en output/0/mesh.glb
OUT = None
for root, dirs, fs in os.walk('output'):
    for f in fs:
        if f.endswith('.glb'):
            OUT = os.path.join(root, f)
print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024/1024, 2)) + ' MB (texturizado)')
      if OUT else '❌ no se generó — copiame el error rojo de arriba')

## Celda 4 — Descargar el `.glb` texturizado
Probalo en https://gltf-viewer.donmccurdy.com (vas a verlo **con colores**, sin pasos extra).

In [ ]:
from google.colab import files
import os
for root, dirs, fs in os.walk('/content/stable-fast-3d/output'):
    for f in fs:
        if f.endswith('.glb'):
            files.download(os.path.join(root, f))

---
### Si algo falla
- **`401` / `gated repo` / `Access to model ... is restricted`** → te faltó aceptar la licencia en https://huggingface.co/stabilityai/stable-fast-3d o el token no es válido → rehacé la Celda 1B.
- **`CUDA out of memory`** → en la Celda 3 cambiá `--texture-resolution 1024` por `512`.
- **Error compilando texture_baker/uv_unwrapper** (Celda 1) → corré `!pip install -U setuptools wheel ninja` y repetí la Celda 1.
- **Sale con el fondo pegado / deforme** → subí un PNG con fondo transparente, personaje de frente y entero.
- **La textura de la espalda es inventada** → normal: SF3D la alucina desde la vista frontal. Si querés la espalda fiel a tu dibujo, usá el Space **MV-Adapter Img2Texture** (https://huggingface.co/spaces/VAST-AI/MV-Adapter-Img2Texture) con tu malla + tu imagen, o la web nuestra con frente+espalda.
- Cualquier error rojo, copiámelo y lo destrabamos. ⚡